In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats as st
from scipy.stats import shapiro, jarque_bera
import warnings
warnings.filterwarnings('ignore')



In [2]:
# Set style for professional visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ============================================================================
# SECTION 1: DATA PREPARATION
# ============================================================================
print("="*70)
print("LINEAR REGRESSION 2.1: FIFA WORLD CUP 2026 GOAL DIFFERENCE PREDICTION")
print("="*70)

LINEAR REGRESSION 2.1: FIFA WORLD CUP 2026 GOAL DIFFERENCE PREDICTION


In [3]:
# Load datasets
full_data = pd.read_csv('full.csv')
external_data = pd.read_csv('external_factors.csv')
teams_data = pd.read_csv('teamsk.csv')
venues_data = pd.read_csv('venuesk.csv')

print("\n[STEP 1] Loading and Inspecting Data")
print(f"Full match data shape: {full_data.shape}")
print(f"External factors shape: {external_data.shape}")
print(f"Teams data shape: {teams_data.shape}")
print(f"Venues data shape: {venues_data.shape}")


[STEP 1] Loading and Inspecting Data
Full match data shape: (104, 68)
External factors shape: (50, 9)
Teams data shape: (48, 12)
Venues data shape: (10, 8)


In [4]:
# ============================================================================
# SECTION 2: FEATURE ENGINEERING - SELECT 8 EXPLANATORY VARIABLES
# ============================================================================
print("\n[STEP 2] Feature Engineering: Selecting 8 Pre-Match Explanatory Variables")
print("-"*70)


[STEP 2] Feature Engineering: Selecting 8 Pre-Match Explanatory Variables
----------------------------------------------------------------------


In [21]:
feature_rationale = {
    'elo_rating_diff': 'Home ELO rating - Away ELO rating (team strength)',
    'home_xG': 'Home team expected goals (offensive capability)',
    'away_xG': 'Away team expected goals (opponent threat)',
    'home_Possession': 'Home team possession % (team control)',
    'travel_distance_km': 'Distance traveled by away team (fatigue)',
    'betting_odds_home': 'Betting odds for home team (market assessment)',
    'pitch_quality': 'Venue pitch quality score (playing conditions)',
    'referee_strictness': 'Referee strictness rating (affects play style)'
}

print("\n8 Explanatory Variables:")
for i, (var, desc) in enumerate(feature_rationale.items(), 1):
    print(f"  {i}. {var:25s} → {desc}")


8 Explanatory Variables:
  1. elo_rating_diff           → Home ELO rating - Away ELO rating (team strength)
  2. home_xG                   → Home team expected goals (offensive capability)
  3. away_xG                   → Away team expected goals (opponent threat)
  4. home_Possession           → Home team possession % (team control)
  5. travel_distance_km        → Distance traveled by away team (fatigue)
  6. betting_odds_home         → Betting odds for home team (market assessment)
  7. pitch_quality             → Venue pitch quality score (playing conditions)
  8. referee_strictness        → Referee strictness rating (affects play style)


In [22]:
# ============================================================================
# SECTION 3: DATA PREPARATION - SAFE MERGING
# ============================================================================
print("\n[STEP 3] Merging Data Sources")
print("-"*70)

# Create base dataset from full.csv - select unique matches
base_data = full_data[[
    'homeSquadName', 'awaySquadName', 'homeScore', 'awayScore',
    'homeSquadId', 'awaySquadId', 'venueId', 'venueName',
    'home_xG', 'away_xG', 'home_Possession'
]].drop_duplicates(subset=['homeSquadName', 'awaySquadName']).reset_index(drop=True)

print(f"Base matches: {len(base_data)}")


[STEP 3] Merging Data Sources
----------------------------------------------------------------------
Base matches: 104


In [24]:
# Merge with external factors by index alignment
if len(external_data) >= len(base_data):
    external_subset = external_data.iloc[:len(base_data), :].reset_index(drop=True)
    base_data = base_data.reset_index(drop=True)
    base_data = pd.concat([base_data, external_subset[['travel_distance_km', 'betting_odds_home', 'referee_strictness']]], axis=1)
    print(f"After external merge: {len(base_data)} matches")
else:
    print(f"Warning: External data ({len(external_data)}) smaller than matches ({len(base_data)})")
    base_data = base_data.iloc[:len(external_data)].reset_index(drop=True)
    base_data = pd.concat([base_data, external_data[['travel_distance_km', 'betting_odds_home', 'referee_strictness']].reset_index(drop=True)], axis=1)


In [25]:
# Merge with teams data (using team name as key)
# Create mapping from team name to ELO rating
teams_elo_map = dict(zip(teams_data['team_name'], teams_data['elo_rating']))

In [26]:
# Map ELO ratings using team names
base_data['home_elo_rating'] = base_data['homeSquadName'].map(teams_elo_map)
base_data['away_elo_rating'] = base_data['awaySquadName'].map(teams_elo_map)

print(f"After teams merge: {len(base_data)} matches")
print(f"  ELO home matched: {base_data['home_elo_rating'].notna().sum()}")
print(f"  ELO away matched: {base_data['away_elo_rating'].notna().sum()}")

After teams merge: 50 matches
  ELO home matched: 24
  ELO away matched: 5


In [27]:
# Merge with venues data (using venue name as key)
# First, get venue names from full_data to match
venues_quality_map = dict(zip(venues_data['venue_name'], venues_data['pitch_quality_score']))

In [28]:
# Map pitch quality using venue names
base_data['pitch_quality'] = base_data['venueName'].map(venues_quality_map)

print(f"After venues merge: {len(base_data)} matches")
print(f"  Pitch quality matched: {base_data['pitch_quality'].notna().sum()}")

After venues merge: 50 matches
  Pitch quality matched: 0


In [29]:
# Final dataset - drop rows with missing values
final_data = base_data[[
    'homeSquadName', 'awaySquadName', 'goal_diff',
    'home_xG', 'away_xG', 'home_Possession',
    'travel_distance_km', 'betting_odds_home', 'referee_strictness',
    'home_elo_rating', 'away_elo_rating', 'pitch_quality'
]].dropna()

print(f"\nFinal dataset: {len(final_data)} matches (after removing missing values)")


Final dataset: 0 matches (after removing missing values)


In [30]:
# ============================================================================
# SECTION 4: FEATURE ENGINEERING - CREATE 8 FEATURES
# ============================================================================
print("\n[STEP 4] Creating 8 Features")
print("-"*70)

# Create the 8 features
final_data['elo_rating_diff'] = final_data['home_elo_rating'] - final_data['away_elo_rating']

# Feature set
X_features = [
    'elo_rating_diff',
    'home_xG',
    'away_xG',
    'home_Possession',
    'travel_distance_km',
    'betting_odds_home',
    'pitch_quality',
    'referee_strictness'
]

X = final_data[X_features].copy()
y = final_data['goal_diff'].copy()

print(f"\nDataset: {len(X)} matches × {len(X_features)} features")
print(f"\nResponse Variable (Goal Difference) Summary:")
print(y.describe())


[STEP 4] Creating 8 Features
----------------------------------------------------------------------

Dataset: 0 matches × 8 features

Response Variable (Goal Difference) Summary:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: goal_diff, dtype: float64


In [31]:
# ============================================================================
# SECTION 5: EXPLORATORY DATA ANALYSIS
# ============================================================================
print("\n" + "="*70)
print("[SECTION 5] EXPLORATORY DATA ANALYSIS")
print("="*70)

print("\n[5.1] Descriptive Statistics")
print("-"*70)
print("\nFeature Statistics:")
print(X.describe())

# Correlation analysis
print("\n[5.2] Correlation Analysis")
print("-"*70)

data_with_y = X.copy()
data_with_y['goal_diff'] = y

corr_matrix = data_with_y.corr()
print("\nCorrelation with Goal Difference:")
print(corr_matrix['goal_diff'].sort_values(ascending=False))


[SECTION 5] EXPLORATORY DATA ANALYSIS

[5.1] Descriptive Statistics
----------------------------------------------------------------------

Feature Statistics:
       elo_rating_diff  home_xG  away_xG  home_Possession  travel_distance_km  \
count              0.0      0.0      0.0              0.0                 0.0   
mean               NaN      NaN      NaN              NaN                 NaN   
std                NaN      NaN      NaN              NaN                 NaN   
min                NaN      NaN      NaN              NaN                 NaN   
25%                NaN      NaN      NaN              NaN                 NaN   
50%                NaN      NaN      NaN              NaN                 NaN   
75%                NaN      NaN      NaN              NaN                 NaN   
max                NaN      NaN      NaN              NaN                 NaN   

       betting_odds_home  pitch_quality  referee_strictness  
count                0.0            0.0        

In [32]:
# ============================================================================
# SECTION 6: VISUALIZATIONS
# ============================================================================
print("\n" + "="*70)
print("[SECTION 6] CREATING VISUALIZATIONS")
print("="*70)

# 6.1 Correlation Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=ax, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix: Features vs Goal Difference', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_correlation_heatmap.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 01_correlation_heatmap.png")
plt.close()


[SECTION 6] CREATING VISUALIZATIONS
✓ Saved: 01_correlation_heatmap.png
